# HerBERT fine-tuning na PolEmo2 (Colab, T4 GPU)

1. Runtime → Change runtime type → **T4 GPU**
2. Opcjonalnie: dodaj `HF_TOKEN` w 🔑 Secrets (szybsze pobieranie z Hubu)
3. Uruchom komórki po kolei

In [ ]:
REPO = "https://github.com/mtorzynska/nlp_language_models_lab.git"
BRANCH = "feature/herbert-finetuning"

!git clone -b $BRANCH $REPO
%cd nlp_language_models_lab

In [ ]:
# Colab ma już torch z CUDA — nie używaj uv sync, doinstaluj tylko brakujące pakiety
!pip install -q datasets transformers peft evaluate scikit-learn loguru pyarrow

In [ ]:
# Opcjonalnie: token HF z sekretów Colab (unika throttlingu przy pobieraniu modelu)
import os
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    print("HF_TOKEN nie ustawiony — kontynuuję anonimowo")

import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'BRAK — zmień runtime na T4'}")

In [ ]:
# Przygotowanie danych (parquet są w .gitignore, generujemy na miejscu)
!python -m src.data.prepare

In [ ]:
# Fine-tuning. Pełny zbiór: ~15-20 min na T4.
# Na próbę: dodaj --max-train 1000 (stratified subsampling) i --epochs 1
!python -m src.models.train --task text

## (Opcjonalnie) Zachowanie checkpointów w Drive
Sesje Colab potrafią się rozłączyć — checkpointy z końca każdej epoki są w `data/*/results/`.
Skopiuj je na Drive, żeby przetrwały po zamknięciu sesji.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/lab2_results
!cp -r data/HerbertForSequenceClassification/results /content/drive/MyDrive/lab2_results/ 2>/dev/null || echo 'brak checkpointów'
!cp -r models /content/drive/MyDrive/lab2_results/ 2>/dev/null || echo 'brak zapisanego modelu'